# Day 1 — MCP Labs (Colab)

S4DS KJSIT. Same flow as the `day1/mcp_*.py` scripts:

1. Connect
2. Create a tiny MCP server
3. See tools
4. One prompt
5. Loop

**Before anything — add your DeepSeek API key:**

1. Get a key at [platform.deepseek.com/api_keys](https://platform.deepseek.com/api_keys)
2. Click the key icon in the left sidebar → Add new secret
3. Name it exactly `DEEPSEEK_API_KEY`
4. Paste the key → toggle **Notebook access** on

Local twin: put the same key in `.env` (see `.env.example`).
Required for Labs MCP 3–4 (agent). Labs 0–2 only need the MCP URL.


## Setup — run this once

Loads `DEEPSEEK_API_KEY` from your Colab secret. Fail here if the secret is missing.


In [ ]:
!pip install -q "smolagents[mcp]>=1.14.0" "mcp>=1.9.0,<2.0.0" "fastmcp>=2.0.0" "openai>=1.0.0"

import os
from google.colab import userdata

os.environ["DEEPSEEK_API_KEY"] = userdata.get("DEEPSEEK_API_KEY")
MODEL_ID = "deepseek-chat"
MCP_URL = "https://personal-mindicatormcp.qbegzg.easypanel.host/mcp"

assert os.environ.get("DEEPSEEK_API_KEY"), "Set Colab secret DEEPSEEK_API_KEY"
print("MODEL:", MODEL_ID)
print("MCP URL:", MCP_URL)
print("SETUP OK")


---
## Lab MCP 0 — Connect

First just connect to Mindicator and call `health_check`.

Matches: `day1/mcp_00_connect.py`

No DeepSeek call here — but keep `DEEPSEEK_API_KEY` set from Setup for later labs.


In [ ]:
from smolagents import MCPClient

mcp_config = {"url": MCP_URL, "transport": "streamable-http"}

with MCPClient(mcp_config, structured_output=True) as tools:
    print(f"Connected. {len(tools)} tool(s):\n")
    for tool in tools:
        first_line = (tool.description or "").strip().split("\n")[0]
        print(f"- {tool.name}: {first_line}")

    by_name = {tool.name: tool for tool in tools}
    print("\nhealth_check:")
    print(by_name["health_check"]())


---
## Lab MCP 1 — Create a tiny MCP server

Same idea as Day 1 `@tool`, but the function lives on a server.

Matches: `day1/mcp_01_tiny_server.py`

In Colab we usually **do not keep a long-running server**. Read / edit the code here; run the `.py` file locally in a second terminal.

Server itself needs no API key. An agent talking to it later needs `DEEPSEEK_API_KEY`.


In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("campus-mess")

@mcp.tool
def get_mess_menu(day: str) -> str:
    """Get the hostel mess menu for a weekday.

    Use this for food / mess / dining questions.

    Args:
        day: Day name, e.g. "Monday" or "Tuesday".
    """
    menu = {
        "monday": "Rajma chawal, salad, curd",
        "tuesday": "Pav bhaji, kheer",
        "wednesday": "Veg biryani, raita",
        "thursday": "Chole bhature",
        "friday": "Masala dosa, sambar",
    }
    return menu.get(day.lower().strip(), f"No menu listed for {day}")

@mcp.tool
def is_mess_open(time_str: str) -> str:
    """Check if the hostel mess is open at a given time.

    Args:
        time_str: Time in HH:MM 24h format, e.g. "13:30".
    """
    try:
        hour = int(time_str.split(":")[0])
    except Exception:
        return "Could not parse time. Use HH:MM, e.g. 13:30."

    if 7 <= hour < 10 or 12 <= hour < 15 or 19 <= hour < 22:
        return f"Mess is OPEN at {time_str}."
    return f"Mess is CLOSED at {time_str}."

url = "http://127.0.0.1:8001/mcp"

print("=" * 70)
print("YOU CREATED YOUR MCP SERVER")
print("=" * 70)
print()
print("Name:   campus-mess")
print("Tools:")
print("  - get_mess_menu(day)     → hostel mess menu for a weekday")
print("  - is_mess_open(time_str) → whether mess is open at HH:MM")
print()
print(f"URL:    {url}")
print("Transport: streamable-http")
print()
print("Plug this URL into an MCPClient:")
print()
print("  from smolagents import MCPClient")
print()
print("  mcp_config = {")
print(f'      "url": "{url}",')
print('      "transport": "streamable-http",')
print("  }")
print()
print("  with MCPClient(mcp_config, structured_output=True) as tools:")
print("      for t in tools:")
print("          print(t.name)")
print()
print("To actually serve it, run day1/mcp_01_tiny_server.py in a local terminal.")
print("(Keep mcp.run(...) commented in Colab.)")
# mcp.run(transport="http", host="127.0.0.1", port=8001)


**Try it:** compare `@mcp.tool` above with `@tool` in Day 1 Lab 4. Same idea — different place the function lives.


---
## Lab MCP 2 — See the tools

Read the Mindicator tool descriptions as if you were the model.

Matches: `day1/mcp_02_see_tools.py`

No DeepSeek call here — agent labs next need `DEEPSEEK_API_KEY` from Setup.


In [ ]:
with MCPClient(mcp_config, structured_output=True) as tools:
    for tool in tools:
        print(f"\n{tool.name}")
        print("-" * len(tool.name))
        print((tool.description or "(no description)").strip())


**Try it:** map each question to a tool —
`Is train 95338 late?` → `get_live_status`,
`Churchgate to Thane` → `find_train_path`,
`Auto fare for 5 km at night` → `get_auto_fare`.


---
## Lab MCP 3 — One prompt

Same ToolCallingAgent as Day 1 Lab 4, but tools come from Mindicator MCP.
Uses **DeepSeek** via `OpenAIModel` + Colab secret `DEEPSEEK_API_KEY`.

Matches: `day1/mcp_03_one_prompt.py`


In [ ]:
from smolagents import MCPClient, OpenAIModel, ToolCallingAgent

model = OpenAIModel(
    model_id=MODEL_ID,
    api_base="https://api.deepseek.com/v1",
    api_key=os.environ["DEEPSEEK_API_KEY"],
)

INSTRUCTIONS = """
You are a Mumbai transit helper using Mindicator MCP tools.

Prefer the specialised tools first. Use execute_sql only when no specialised
tool fits.

How to choose tools:
- health_check: confirm the server + DB are up
- get_schema: list tables/columns before writing custom SQL
- execute_sql: read-only SELECT/WITH fallback for odd queries (LIMIT capped)
- get_live_status: live running status for a suburban train number (e.g. 95338)
- search_stations: find stations by name
- find_train_path: path hints between two stations (e.g. Churchgate → Thane)
- get_ticket_fare: suburban OD ticket fares between two stations
- search_bus_routes: find bus routes by code/agency
- get_bus_route_stops: ordered stops on a bus route (e.g. BEST 1(Up))
- get_auto_fare: auto rickshaw day/night fare by km

Rules:
- Prefer real tool results over guessing.
- Prefer find_train_path / get_ticket_fare / get_auto_fare / get_bus_route_stops
  over inventing SQL when those tools exist.
- Keep any SQL simple. Use LIMIT. Station names are often UPPERCASE in the DB
  (e.g. CHURCHGATE, THANE, DADAR).
- Answer in a few short sentences for a student.
- If the tools cannot answer, say so clearly.
""".strip()


TASK = "How do I get from Churchgate to Thane on the local train?"

with MCPClient(mcp_config, structured_output=True) as tools:
    agent = ToolCallingAgent(
        tools=tools,
        model=model,
        max_steps=8,
        instructions=INSTRUCTIONS,
    )
    print(agent.run(TASK))


**Try it:** `Ticket fare from Churchgate to Thane`, `Auto fare for 5 km at night`, `Stops on BEST bus route 1(Up)`, `Is train 95338 running late right now?`


---
## Lab MCP 4 — Loop

Same chat loop as `day1/mcp_04_loop.py`.
Type a question, get an answer, repeat. Type `quit` to stop.
Uses **DeepSeek** (`DEEPSEEK_API_KEY` from Setup).

Matches: `day1/mcp_04_loop.py`


In [ ]:
with MCPClient(mcp_config, structured_output=True) as tools:
    agent = ToolCallingAgent(
        tools=tools,
        model=model,
        max_steps=8,
        instructions=INSTRUCTIONS,
    )

    print("Ask about Mumbai trains, buses, fares, or a live train number.")
    print("Type quit to stop.")
    print()

    while True:
        question = input("You: ").strip()
        if question.lower() in {"quit", "exit", "q"}:
            print("Bye.")
            break
        if not question:
            continue

        print()
        print("Agent:")
        print(agent.run(question))
        print()


---
## Project 2

Build your own version following:

1. Connect
2. (Optional) create a tiny MCP server
3. See tools
4. One prompt
5. Loop

Keep `DEEPSEEK_API_KEY` in Colab secrets (or `.env` locally). See `projects/project-2.md`.
